# Pipeline de análisis DEI en Participedia

Este notebook ejecuta el pipeline completo de detección y clasificación de categorías DEI sobre los casos de Participedia:

1. Carga de datos desde la base de datos
2. Detección de categorías DEI mediante expresiones regulares
3. Clasificación de grupos y roles mediante LLM
4. Almacenamiento de resultados en la base de datos

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import pandas as pd
from sqlalchemy import text

from modules.config import engine
from modules.patterns import dei_patterns
from modules.analysis import extraer_palabras_exactas, get_continent
from modules.llm import clasificar_rol_y_grupos_openai

## 1. Carga de datos

In [ ]:
df_cases = pd.read_sql("SELECT * FROM cases", engine)

## 2. Construcción del texto combinado

Se combinan la descripción, los participantes objetivo y todas las secciones del cuerpo del caso en un único campo de texto para el análisis.

In [ ]:
columnas_candidatas = ['description', 'targeted_participants'] + \
                      [col for col in df_cases.columns if col.startswith('section_')]

df_cases['texto_combinado'] = df_cases[columnas_candidatas].fillna('').apply(
    lambda row: ' '.join(row.values), axis=1
)

## 3. Detección de categorías DEI mediante expresiones regulares

In [ ]:
df_words = df_cases.apply(
    lambda row: extraer_palabras_exactas(row, columnas_candidatas), axis=1
).apply(pd.Series)

df = pd.concat([df_cases[['id']], df_words], axis=1)

for cat in dei_patterns:
    df[cat] = df[f"source_{cat}"].notna().astype(int)

## 4. Almacenamiento de métricas DEI en la base de datos

In [ ]:
cols_orden = (
    ['id'] +
    [cat for cat in dei_patterns] +
    [f"source_{cat}" for cat in dei_patterns]
)
df_final = df[cols_orden]

df_final.to_sql('cases_dei_metrics_new', engine, if_exists='replace', index=False)

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS cases_dei_metrics_backup"))
    conn.execute(text("ALTER TABLE IF EXISTS cases_dei_metrics RENAME TO cases_dei_metrics_backup"))
    conn.execute(text("ALTER TABLE cases_dei_metrics_new RENAME TO cases_dei_metrics"))
    conn.commit()

## 5. Enriquecimiento geográfico: país → continente

In [ ]:
with engine.connect() as conn:
    for _, row in df_cases.iterrows():
        continent = get_continent(row['country'])
        conn.execute(
            text("UPDATE cases SET continent = :continent WHERE id = :id"),
            {"continent": continent, "id": row['id']}
        )
    conn.commit()

## 6. Clasificación de grupos y roles mediante LLM

Se procesan únicamente los casos con al menos una categoría DEI detectada (1.518 casos).
Los resultados se guardan en un archivo Excel tras cada caso para permitir reanudar el proceso si se interrumpe.

In [ ]:
ARCHIVO_SALIDA = 'resultados_llm.xlsx'
ESPERA = 0.3

def get_cats_detectadas(row):
    return [cat for cat in dei_patterns if row.get(cat, 0) == 1]

df['cats_detectadas'] = df.apply(get_cats_detectadas, axis=1)
df_a_procesar = df[df['cats_detectadas'].apply(len) > 0].copy()

df_resultados = df_a_procesar[['id', 'cats_detectadas']].copy()
df_resultados['llm_clasificacion'] = None

ids_a_procesar = df_resultados['id'].tolist()
print(f"Casos a procesar: {len(ids_a_procesar)}")

In [ ]:
for i, case_id in enumerate(ids_a_procesar):
    try:
        row = df.loc[df['id'] == case_id].iloc[0]
        row_cases = df_cases.loc[df_cases['id'] == case_id].iloc[0]
        source_data = df.loc[df['id'] == case_id].iloc[0].to_dict()

        resultado = clasificar_rol_y_grupos_openai(
            case_id,
            row_cases['texto_combinado'],
            row['cats_detectadas'],
            row_cases=row_cases,
            source_data=source_data
        )

        df_resultados.loc[df_resultados['id'] == case_id, 'llm_clasificacion'] = resultado
        df_resultados.to_excel(ARCHIVO_SALIDA, index=False)

    except Exception as e:
        print(f"[!] Error caso {case_id}: {e}")

    if (i + 1) % 100 == 0:
        print(f"Procesados {i + 1}/{len(ids_a_procesar)}")

    time.sleep(ESPERA)

pendientes = df_resultados['llm_clasificacion'].isna().sum()
print(f"Completado. Pendientes: {pendientes}")

## 7. Almacenamiento de resultados del LLM en la base de datos

In [ ]:
df_resultados.to_sql('cases_llm_classification', engine, if_exists='replace', index=False)